[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_23/2_sql.ipynb)

# Day 23: SQL (hard)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (26 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Typical track length per genre | hard | 5 |
| Q2 | Order value percentiles, UK versus abroad | hard | 6 |
| Q3 | How long from first look to first purchase? (review) | hard | 6 |
| Q4 | How concentrated is the rating activity? | hard | 5 |
| Q5 | Order count histogram per platform (review) | hard | 4 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: Online Retail** (UCI, a UK online gift shop, 2010-12-01 to 2011-12-09, CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (text; a 'C' prefix = cancellation), `stock_code`, `description`, `quantity` (negative for cancellations), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price` (GBP), `customer_id` (NULL for about 25% of rows), `country` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5.0 in steps of 0.5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title`, `year` (from the title, can be NULL), `genres` (pipe separated, e.g. 'Action|Comedy') |
| `ml_movie_genres` | one (movie, genre) pair | `movie_id`, `genre` (the `genres` string split into rows) |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (a UK gift shop, 2010-12 to 2011-12, CC BY 4.0).
import io, zipfile
_p = os.path.join(DATA, "online_retail.csv.gz")
if not os.path.exists(_p):                       # first run in Colab: download the UCI zip and convert it once
    with zipfile.ZipFile(data_path("online_retail.zip", "https://archive.ics.uci.edu/static/public/352/online+retail.zip")) as zf:
        pd.read_excel(io.BytesIO(zf.read("Online Retail.xlsx"))).to_csv(_p, index=False, compression="gzip")
_o = pd.read_csv(_p, dtype={"InvoiceNo": str, "StockCode": str})
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o.to_sql("retail", db, index=False)

# Real data: MovieLens small (GroupLens, research and education use). Downloads the zip if the CSVs are missing.
import zipfile
if not os.path.exists(os.path.join(DATA, "movielens_ratings.csv")):
    _z = data_path("ml-latest-small.zip", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip")
    with zipfile.ZipFile(_z) as zf:
        for _n in ("ratings", "movies"):
            with open(os.path.join(DATA, f"movielens_{_n}.csv"), "wb") as fh:
                fh.write(zf.read(f"ml-latest-small/{_n}.csv"))
_r = pd.read_csv(data_path("movielens_ratings.csv"))
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r = _r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]]
_r.to_sql("ml_ratings", db, index=False)
_m = pd.read_csv(data_path("movielens_movies.csv")).rename(columns={"movieId": "movie_id"})
_m["year"] = pd.to_numeric(_m["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce").astype("Int64")
_m[["movie_id", "title", "year", "genres"]].to_sql("ml_movies", db, index=False)
_g = _m.assign(genre=_m["genres"].str.split("|")).explode("genre")[["movie_id", "genre"]]
_g[_g["genre"] != "(no genres listed)"].to_sql("ml_movie_genres", db, index=False)

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Typical track length per genre

*hard, about 5 min*

Chinook (real). SQLite (like many exam editors) has no `MEDIAN` function. For every genre with at
least 20 tracks return `genre`, `tracks`, `median_min` (median of `Milliseconds / 60000.0`, 2 decimals) and `avg_min`
(2 decimals). Show the 6 genres with the highest median.

Example: lengths 3, 4, 10 have median 4; lengths 3, 4, 5, 10 have median 4.5.

*Follow-up:* for TV Shows the average is far below the median. What does that tell you about the data?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'median' without a MEDIAN function. Pattern: median with ROW_NUMBER and COUNT windows: keep the middle row (odd count) or the two middle rows (even count) and average them.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `ROW_NUMBER() OVER (PARTITION BY genre ORDER BY ms)` and `COUNT(*) OVER (PARTITION BY genre)`.
2. Middle rows: `rn IN ((cnt + 1) / 2, (cnt + 2) / 2)` (integer division: one row if cnt is odd, two if even).
3. AVG of those rows per genre; join the plain AVG per genre for comparison.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH t AS (
  SELECT g.Name AS genre, t.Milliseconds / 60000.0 AS minutes,
    ROW_NUMBER() OVER (PARTITION BY t.GenreId ORDER BY t.Milliseconds) AS rn,
    COUNT(*) OVER (PARTITION BY t.GenreId) AS cnt
  FROM Track t JOIN Genre g ON g.GenreId = t.GenreId
), med AS (
  SELECT genre, cnt AS tracks, AVG(minutes) AS median_min
  FROM t
  WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
  GROUP BY genre, cnt
), avgs AS (
  SELECT genre, AVG(minutes) AS avg_min FROM t GROUP BY genre
)
SELECT m.genre, m.tracks, ROUND(m.median_min, 2) AS median_min, ROUND(a.avg_min, 2) AS avg_min
FROM med m JOIN avgs a ON a.genre = m.genre
WHERE m.tracks >= 20
ORDER BY m.median_min DESC
LIMIT 6
```

Result:

| genre | tracks | median_min | avg_min |
|---|---|---|---|
| Sci Fi & Fantasy | 26 | 48.73 | 48.53 |
| Drama | 64 | 43.5 | 42.92 |
| TV Shows | 93 | 43.03 | 35.75 |
| Electronica/Dance | 30 | 5.04 | 5.05 |
| Heavy Metal | 28 | 4.91 | 4.96 |
| Metal | 374 | 4.74 | 5.16 |

**Why:** `(cnt + 1) / 2` and `(cnt + 2) / 2` are the same row when cnt is odd (cnt = 5: 3 and 3) and the two middle rows when it is even (cnt = 4: 2 and 3), so one formula covers both cases. Follow-up: TV Shows has median 43.03 but mean 35.75: the distribution is skewed LEFT, a group of short items (trailers or short episodes) pulls the mean down. That is why medians are preferred for skewed metrics like watch time.

**Complexity:** One window sort per genre: O(n log n).

**Common mistakes:** `rn = cnt / 2` only (wrong for odd counts, or picks one of the two middle rows for even counts). Integer division of Milliseconds by 60000. PostgreSQL has `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY x)`; BigQuery has `APPROX_QUANTILES`; say so, then write the portable version.

**Learn more:** [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q2. Order value percentiles, UK versus abroad

*hard, about 6 min*

UCI Online Retail (real). An order is one `invoice_no`; its value is `SUM(quantity * unit_price)`. Use
only real sales: `invoice_no` not starting with 'C', `quantity > 0`, `unit_price > 0`. Split orders into `market` =
'UK' (country is United Kingdom) or 'other'. Return per market: `n` orders, `p50`, `p90`, `p99` (nearest-rank method:
the smallest value whose cumulative share is at least p) and `mean`, all values with 2 decimals.

Example (nearest rank): values 10, 20, 30, 40: p50 = 20 (2 of 4 = 50% are <= 20), p90 = 40.

*Follow-up:* the mean is far above the median in both markets. Which number goes on the dashboard, and why?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: p90 / p99 per group. Pattern: percentiles with CUME_DIST (or ROW_NUMBER / COUNT), then `MIN(CASE WHEN cume_dist >= p THEN value END)` and a pivot of p50, p90, p99 into columns.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `inv`: GROUP BY invoice_no, market: SUM(quantity * unit_price).
2. CTE `cd`: `CUME_DIST() OVER (PARTITION BY market ORDER BY value)`.
3. Per market: `MIN(CASE WHEN c >= 0.9 THEN value END)` etc.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH inv AS (
  SELECT invoice_no,
    CASE WHEN country = 'United Kingdom' THEN 'UK' ELSE 'other' END AS market,
    SUM(quantity * unit_price) AS value
  FROM retail
  WHERE invoice_no NOT LIKE 'C%' AND quantity > 0 AND unit_price > 0
  GROUP BY invoice_no, market
), cd AS (
  SELECT market, value,
    CUME_DIST() OVER (PARTITION BY market ORDER BY value) AS c
  FROM inv
)
SELECT market, COUNT(*) AS n,
  ROUND(MIN(CASE WHEN c >= 0.50 THEN value END), 2) AS p50,
  ROUND(MIN(CASE WHEN c >= 0.90 THEN value END), 2) AS p90,
  ROUND(MIN(CASE WHEN c >= 0.99 THEN value END), 2) AS p99,
  ROUND(AVG(value), 2) AS mean
FROM cd
GROUP BY market
ORDER BY market
```

Result:

| market | n | p50 | p90 | p99 | mean |
|---|---|---|---|---|---|
| UK | 18019 | 300.5 | 869.72 | 4406.61 | 500.87 |
| other | 1941 | 424.06 | 1516.68 | 9341.26 | 845.68 |

**Why:** CUME_DIST is the share of rows with a value less than or equal to the current one, so the first value that reaches p is the nearest-rank percentile. Ties get the same CUME_DIST, so tied values never split. UK: p50 300.50, mean 500.87; abroad: p50 424.06, p99 9,341.26. Follow-up: show the median (typical order) and p90 (big orders) on the dashboard; the mean is pulled up by a few wholesale orders and moves when one huge order arrives. Report the mean only when you need totals (revenue = mean * orders).

**Complexity:** Aggregation, then one window sort per market: O(n log n).

**Common mistakes:** Grouping by country without the CASE (one row per country). Taking percentiles of LINE values instead of ORDER values. Using NTILE(100) for p99 (NTILE splits rows into equal buckets and breaks ties). PostgreSQL: `PERCENTILE_DISC(0.9) WITHIN GROUP (ORDER BY value)` is exactly nearest rank.

**Learn more:** [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot)

</details>

---
### Q3. How long from first look to first purchase? (review)

*hard, about 6 min*

Shopping app (made up). For each user take the first `view_item` time and the first `purchase` **after**
it (`app_events`). For users who have both, compute the gap in hours. Return per `platform` (from `app_users`):
`buyers` (users with both events) and `median_hours` (1 decimal).

Example: first view Monday 10:00, purchases Monday 09:00 and Wednesday 10:00: the gap is 48 hours (the Monday 09:00
purchase is before the first view, so it does not count).

*Follow-up:* the average gap is about 2 to 2.5 times the median. Why report the median here?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: time between two funnel steps, 'median'. Pattern: funnel step pairs (first A, first B after A) plus a median via ROW_NUMBER / COUNT.

</details>

<details><summary><b>Hint 2</b></summary>

1. `fv`: MIN(event_time) of view_item per user.
2. `fp`: join purchases with `event_time > view_t`, MIN per user.
3. hours = `(julianday(buy_t) - julianday(view_t)) * 24`, join platform.
4. Median per platform with `rn IN ((cnt + 1) / 2, (cnt + 2) / 2)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH fv AS (
  SELECT user_id, MIN(event_time) AS view_t
  FROM app_events WHERE event_name = 'view_item'
  GROUP BY user_id
), fp AS (
  SELECT fv.user_id, fv.view_t, MIN(e.event_time) AS buy_t
  FROM fv
  JOIN app_events e
    ON e.user_id = fv.user_id AND e.event_name = 'purchase' AND e.event_time > fv.view_t
  GROUP BY fv.user_id, fv.view_t
), d AS (
  SELECT u.platform, (julianday(fp.buy_t) - julianday(fp.view_t)) * 24 AS hours
  FROM fp JOIN app_users u ON u.user_id = fp.user_id
), r AS (
  SELECT platform, hours,
    ROW_NUMBER() OVER (PARTITION BY platform ORDER BY hours) AS rn,
    COUNT(*) OVER (PARTITION BY platform) AS cnt
  FROM d
)
SELECT platform, cnt AS buyers, ROUND(AVG(hours), 1) AS median_hours
FROM r
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY platform, cnt
ORDER BY platform
```

Result:

| platform | buyers | median_hours |
|---|---|---|
| android | 205 | 85.6 |
| ios | 176 | 115.3 |
| web | 41 | 134.1 |

**Why:** The join condition `e.event_time > fv.view_t` makes the steps ordered, and MIN picks the first qualifying purchase. Android buyers convert fastest (median 85.6 hours), web slowest (134.1). Follow-up: the means are 212.5 (android), 230.9 (ios) and 260.2 (web) hours: a long tail of users who buy weeks later pulls the mean up. The median describes the typical buyer and is robust to that tail.

**Complexity:** Two aggregations on events plus a window sort: O(n log n).

**Common mistakes:** Using the first purchase overall (it can be before the first view: negative gaps). Subtracting text timestamps directly. In PostgreSQL use `EXTRACT(EPOCH FROM buy_t - view_t) / 3600`.

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles)

</details>

---
### Q4. How concentrated is the rating activity?

*hard, about 5 min*

MovieLens (real). Count ratings per user. Return one row with: `median_n` (median ratings per user),
`p90_n` (nearest-rank p90), `mean_n` (1 decimal) and `top10_share`: the % of ALL ratings made by the 10% of users who
rate most (1 decimal).

Example: 10 users with counts 1, 1, 1, 1, 1, 1, 1, 1, 1, 91: median 1, top 10% (one user) share = 91 / 100 = 91.0%.

*Follow-up:* a PM asks for "the average user". What do you answer?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'median', 'p90' and 'top 10% of users'. Pattern: percentiles with ROW_NUMBER / CUME_DIST, and NTILE(10) for deciles.

</details>

<details><summary><b>Hint 2</b></summary>

1. `per_user`: COUNT(*) per user_id.
2. One CTE with ROW_NUMBER, COUNT(*) OVER (), CUME_DIST and `NTILE(10) OVER (ORDER BY n DESC)`.
3. Median: AVG over the middle rows (scalar subquery); p90: `MIN(CASE WHEN cd >= 0.9 THEN n END)`; share: `SUM(CASE WHEN decile = 1 THEN n END) / SUM(n)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH per_user AS (
  SELECT user_id, COUNT(*) AS n FROM ml_ratings GROUP BY user_id
), r AS (
  SELECT n,
    ROW_NUMBER() OVER (ORDER BY n) AS rn,
    COUNT(*) OVER () AS cnt,
    CUME_DIST() OVER (ORDER BY n) AS cd,
    NTILE(10) OVER (ORDER BY n DESC) AS decile
  FROM per_user
)
SELECT
  (SELECT AVG(n) FROM r WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)) AS median_n,
  MIN(CASE WHEN cd >= 0.9 THEN n END) AS p90_n,
  ROUND(AVG(n), 1) AS mean_n,
  ROUND(100.0 * SUM(CASE WHEN decile = 1 THEN n ELSE 0 END) / SUM(n), 1) AS top10_share
FROM r
```

Result:

| median_n | p90_n | mean_n | top10_share |
|---|---|---|---|
| 70.5 | 400 | 165.3 | 47.7 |

**Why:** NTILE(10) over users sorted by activity gives deciles of USERS (61 users each for 610 users), and summing their ratings gives their share. The typical user has 70.5 ratings, the mean is 165.3, and the top 10% of users make 47.7% of all ratings. Follow-up: there is no 'average user' in such a skewed distribution: give the median (70.5), and say that the top decile behaves very differently, so models and metrics should be checked per segment (heavy versus light users).

**Complexity:** One aggregation and one window sort over users.

**Common mistakes:** NTILE over RATINGS instead of users. Taking the mean as 'typical'. Forgetting that NTILE puts the extra rows in the first buckets when n is not divisible by 10 (fine here, but say it).

**Learn more:** [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q5. Order count histogram per platform (review)

*hard, about 4 min*

Shopping app (made up). Count **completed** orders per user (`purchases`), keeping users with zero. Return
one row per `platform` with columns `users`, `o0`, `o1`, `o2`, `o3plus` (number of users with 0, 1, 2, 3 or more
completed orders) and `repeat_pct` = users with 2+ orders / users with 1+ orders (1 decimal). Order by platform.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: buckets as columns. Pattern: pivot with conditional aggregation on a per-user count built with a LEFT JOIN (status filter in the ON clause).

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `app_users u LEFT JOIN purchases p ON p.user_id = u.user_id AND p.status = 'completed'`, COUNT(p.order_id) per user.
2. Outer: `SUM(CASE WHEN orders = 0 THEN 1 ELSE 0 END)` etc. per platform.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH n AS (
  SELECT u.user_id, u.platform, COUNT(p.order_id) AS orders
  FROM app_users u
  LEFT JOIN purchases p ON p.user_id = u.user_id AND p.status = 'completed'
  GROUP BY u.user_id, u.platform
)
SELECT platform, COUNT(*) AS users,
  SUM(CASE WHEN orders = 0 THEN 1 ELSE 0 END) AS o0,
  SUM(CASE WHEN orders = 1 THEN 1 ELSE 0 END) AS o1,
  SUM(CASE WHEN orders = 2 THEN 1 ELSE 0 END) AS o2,
  SUM(CASE WHEN orders >= 3 THEN 1 ELSE 0 END) AS o3plus,
  ROUND(100.0 * SUM(CASE WHEN orders >= 2 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN orders >= 1 THEN 1 ELSE 0 END), 0), 1) AS repeat_pct
FROM n
GROUP BY platform
ORDER BY platform
```

Result:

| platform | users | o0 | o1 | o2 | o3plus | repeat_pct |
|---|---|---|---|---|---|---|
| android | 695 | 503 | 82 | 56 | 54 | 57.3 |
| ios | 594 | 422 | 72 | 46 | 54 | 58.1 |
| web | 211 | 172 | 18 | 7 | 14 | 53.8 |

**Why:** The status filter sits in the ON clause, so users whose only orders were refunded still appear with 0. COUNT(p.order_id) counts only matched rows (COUNT(*) would give 1 to non-buyers). Repeat rates are close (53.8% to 58.1%), so platforms differ in how many users buy at all, not in loyalty once they buy.

**Complexity:** One join and two GROUP BYs: O(users + orders).

**Common mistakes:** `WHERE p.status = 'completed'` after a LEFT JOIN (drops non-buyers: o0 becomes 0). COUNT(*) instead of COUNT(p.order_id). Repeat rate over ALL users instead of buyers.

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_23/3_stats.ipynb)